# Phân tích kết quả
Đọc CSV do evaluate/ablation xuất. Không tạo số liệu giả khi chưa chạy thí nghiệm.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/pipeline.py").exists()), None)
if ROOT is None:
    raise RuntimeError("Đặt working directory trong project đã giải nén/clone.")
os.chdir(ROOT)
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import cv2
import matplotlib.pyplot as plt


In [ ]:
import pandas as pd
TABLES = Path("report/tables")
files = sorted(TABLES.glob("*.csv"))
print("Available CSV:", [p.name for p in files])
EVALUATION = TABLES/"evaluation.csv"
if not EVALUATION.exists():
    EVALUATION = TABLES/"smoke_shadow.csv"
    print("Fallback: kết quả smoke test, không phải benchmark.")
results = pd.read_csv(EVALUATION) if EVALUATION.exists() else pd.DataFrame()
if results.empty:
    print("Chạy python -m scripts.evaluate trước, rồi sửa EVALUATION nếu cần.")
else:
    display(results.head())
    columns = [c for c in ["psnr","ssim","ms_ssim","ld_map_epe_px","cer","wer","elapsed_seconds"] if c in results]
    display(results.groupby("configuration")[columns].agg(["mean","std","count"]))
    print("Không so sánh số đo smoke_only với mô hình đã train đầy đủ.")


In [ ]:
if not results.empty:
    metrics = [c for c in ["psnr", "ssim", "cer", "elapsed_seconds"] if c in results and results[c].notna().any()]
    if metrics:
        fig, axes = plt.subplots(1,len(metrics),figsize=(5*len(metrics),4),squeeze=False)
        for ax,metric in zip(axes.flat,metrics):
            results.groupby("configuration")[metric].mean().plot.bar(ax=ax,title=metric,rot=35)
        plt.tight_layout()
        fig.savefig("report/figures/pipeline_metrics.png",dpi=160)


In [ ]:
ABLATION = TABLES/"ablation.csv"
if not ABLATION.exists(): ABLATION = TABLES/"smoke_ablation.csv"
if ABLATION.exists():
    ablation = pd.read_csv(ABLATION)
    numeric = ablation.select_dtypes(include="number").columns
    display(ablation.groupby("configuration")[numeric].mean())
else:
    print("Chưa có ablation.csv; chạy scripts.run_ablation trên cùng tập test.")


## Kiểm tra trước khi viết kết luận
- PSNR/SSIM cần cặp cùng tọa độ; ảnh thật chỉ có text chuẩn thì dùng CER/WER.
- Ghi số mẫu thất bại, không loại ảnh khó sau khi xem test.
- `ld_map_epe_px` là map endpoint error, không phải DocUNet LD chính thức.
- Lưu cấu hình, seed, checkpoint, nguồn dữ liệu và phần cứng.
- Đánh giá thủ công nét giả, mất dấu tiếng Việt và chữ bị nhị phân hóa xóa mất.